# Project 26 — Table + Text Local RAG

## Combine CSVs and Text Documents in One Pipeline

**Goal:** Build a unified retrieval system that handles both structured
data (CSV tables) and unstructured text, with source attribution.

**Stack:** Ollama · LlamaIndex · pandas · Jupyter

### What You'll Learn
1. Convert tabular data to searchable text
2. Build a unified index across tables and documents
3. Answer numerical and narrative questions
4. Source attribution (data vs narrative)

### Prerequisites
- **Ollama** running with `nomic-embed-text` and `qwen3:8b`

In [ ]:
!pip install -q llama-index llama-index-llms-ollama llama-index-embeddings-ollama pandas

## Step 1 — Verify Ollama Is Running

In [ ]:
import requests
OLLAMA_BASE = "http://localhost:11434"
try:
    r = requests.get(f"{OLLAMA_BASE}/api/tags", timeout=5)
    r.raise_for_status()
    print(f"Ollama running — {len(r.json().get('models',[]))} model(s)")
except Exception as e:
    print(f"Cannot reach Ollama: {e}")

## Step 2 — Configure LlamaIndex

In [ ]:
from llama_index.core import Settings, VectorStoreIndex, Document
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding
import pandas as pd

Settings.llm = Ollama(model="qwen3:8b", request_timeout=120.0)
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")
print("LlamaIndex configured")

## Step 3 — Create Mixed Data Sources

In [ ]:
sales = pd.DataFrame({
    "Quarter": ["Q1 2024", "Q2 2024", "Q3 2024", "Q4 2024"],
    "Revenue_M": [45.2, 52.1, 58.7, 67.3],
    "Units_Sold": [12000, 14500, 16200, 19800],
    "Top_Product": ["Widget Pro", "Widget Pro", "Widget Max", "Widget Max"],
})
print("Sales Data:")
print(sales.to_string(index=False))

narrative = """Annual Business Review 2024

Our company achieved record revenue of $223.3M in 2024, 35% YoY growth.
Widget Max launched in Q3, exceeded expectations in Asia Pacific.

Key wins: Fortune 500 deal Q2 ($8M ARR), government contract Q4 ($12M/3yr).
Sales team expanded from 45 to 72 people.

Challenges: Supply chain disruptions in Q1. Customer churn rose to 8%.
We responded with a loyalty program in Q3.

2025 priorities: expand to Latin America, launch Widget Ultra,
reduce churn below 4%, reach $300M revenue target."""
print(f"\nNarrative: {len(narrative)} chars")

## Step 4 — Build Combined Index

In [ ]:
table_docs = []
for _, row in sales.iterrows():
    text = (f"In {row['Quarter']}, revenue was ${row['Revenue_M']}M with "
            f"{row['Units_Sold']:,} units sold. Top product: {row['Top_Product']}.")
    table_docs.append(Document(text=text,
        metadata={"source": "sales_table", "type": "data", "quarter": row["Quarter"]}))

# Summary statistics
total_rev = sales["Revenue_M"].sum()
summary = (f"Full year 2024: Revenue ${total_rev:.1f}M, "
           f"{sales['Units_Sold'].sum():,} units sold. "
           f"Best quarter: Q4 with ${sales['Revenue_M'].max()}M.")
table_docs.append(Document(text=summary,
    metadata={"source": "sales_summary", "type": "data"}))

text_doc = Document(text=narrative,
    metadata={"source": "business_review", "type": "narrative"})

all_docs = table_docs + [text_doc]
index = VectorStoreIndex.from_documents(all_docs, show_progress=True)
engine = index.as_query_engine(similarity_top_k=4)
print(f"\nCombined index: {len(all_docs)} documents")

## Step 5 — Query Across Tables and Text

In [ ]:
queries = [
    ("What was the total revenue in 2024?", "data"),
    ("Which product performed best in Q4?", "data"),
    ("What were the main challenges this year?", "narrative"),
    ("What is the 2025 revenue target?", "narrative"),
]

for q, expected_type in queries:
    response = engine.query(q)
    sources = [n.metadata.get("source", "?") for n in response.source_nodes]
    print(f"\nQ: {q}")
    print(f"A: {str(response)[:150]}")
    print(f"Sources: {sources}")

## Limitations

1. **Table-to-text lossy:** Converting tables to prose may lose precision.
2. **No SQL fallback:** Numerical aggregations are better served by SQL.
3. **Scale:** Row-per-document works only for small tables.

## What You Learned

| Concept | What We Did |
|---|---|
| **Table-to-text** | CSV rows as natural language |
| **Summary stats** | Added computed aggregates |
| **Unified index** | Single vector store for mixed data |
| **Source attribution** | Tracked data vs narrative provenance |